# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by their `@id` values.

In [ ]:
# List all available record sets, fields, and columns with their @id for reference.
print('Record Sets:')
record_sets = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    for rs in metadata.record_set:
        print(f"- @id: {getattr(rs, '@id', None)}, name: {getattr(rs, 'name', None)}")
        record_sets.append(getattr(rs, '@id', None))

    # For each record set, print its fields and columns
    for rs in metadata.record_set:
        print(f"\nFields for Record Set @id: {getattr(rs, '@id', None)}")
        if hasattr(rs, 'field') and rs.field:
            for fld in rs.field:
                print(f"  Field @id: {getattr(fld, '@id', None)}, name: {getattr(fld, 'name', None)}, dataType: {getattr(fld, 'data_type', None)}")
                if hasattr(fld, 'column') and fld.column:
                    print(f"    Columns: {[getattr(c, '@id', None) for c in (fld.column if isinstance(fld.column, list) else [fld.column])]}")
else:
    print('No record sets defined in the Croissant schema.')

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All references use their `@id` values.

In [ ]:
# Set up the record_set @id(s) for data extraction - update with discovered record set @id(s):
## If no record sets are defined, try the most common pattern - first distribution file as the main table
main_record_set = None
if hasattr(metadata, 'record_set') and metadata.record_set:
    main_record_set = getattr(metadata.record_set[0], '@id', None)
    record_sets_ids = [getattr(rs, '@id', None) for rs in metadata.record_set]
else:
    # For datasets with no explicit record_set, check for a DataFileObject/distribution to construct the @id
    record_sets_ids = []
    if hasattr(metadata, 'distribution') and metadata.distribution:
        for dist in metadata.distribution:
            if hasattr(dist, '@id'):
                record_sets_ids.append(dist['@id'] if isinstance(dist, dict) else getattr(dist, '@id', None))
    if record_sets_ids:
        main_record_set = record_sets_ids[0]

print('Attempting to extract the following record set IDs:')
print(record_sets_ids)

dataframes = {}
for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# For demonstration, print the first rows of the main table
if main_record_set and main_record_set in dataframes:
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical processing and filtering for one major numeric field in the main table. All entities are referenced by their `@id` as per the schema.

In [ ]:
# Choose a numeric field
df = dataframes.get(main_record_set)
if df is not None:
    # Try to automatically pick a numeric column for demonstration
    numeric_fields = df.select_dtypes(include=['int', 'float']).columns.tolist()
    print('Detected numeric fields:', numeric_fields)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use the first numeric column
        print(f'Using numeric field (potentially @id): {numeric_field_id}')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fiu' else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {filtered_df.shape[0]} records")
        # Normalize and display
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())
        # Try grouping by the first string/categorical column
        group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            print(f'Grouping by field: {group_field_id}')
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(grouped_df.head())
        else:
            print('No categorical fields found for grouping.')
    else:
        print('No numeric fields available for EDA.')
else:
    print('Main dataframe not loaded. Skipping EDA.')

## 5. Visualization
Visualize distributions of one numeric field, and optionally groupings if data permits.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and 'numeric_field_id' in locals() and numeric_field_id in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id exists, plot mean by group
    if 'group_field_id' in locals() and group_field_id in df:
        plt.figure(figsize=(8,4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.xticks(rotation=90)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()

## 6. Conclusion
In this notebook, we loaded metadata and data from the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, reviewed the available record sets, fields, and columns by their `@id`, performed numeric analysis and transformations, and visualized data distributions. All steps referenced entities using their Croissant schema `@id` fields, facilitating reproducible and robust FAIR data exploration.

**Next steps:** You can further extend this notebook by deepening your statistical analysis, joining multiple record sets, or applying your own hypothesis-driven analyses leveraging the explicit data structure provided by Croissant metadata.